In [1]:
import os
import pandas as pd
import taxonomia

pd.set_option('display.max_columns', None)

In [2]:
def obtener_paths_cantos(base_dir):
    data = []

    # Recorrer las familias
    for familia in os.listdir(base_dir):
        familia_path = os.path.join(base_dir, familia)
        if os.path.isdir(familia_path):
            # Recorrer los géneros
            for genero in os.listdir(familia_path):
                genero_path = os.path.join(familia_path, genero)
                if os.path.isdir(genero_path):
                    # Recorrer las especies
                    for especie in os.listdir(genero_path):
                        especie_path = os.path.join(genero_path, especie)
                        if os.path.isdir(especie_path):
                            # Recorrer los nombres comunes
                            for nombre_comun in os.listdir(especie_path):
                                nombre_comun_path = os.path.join(especie_path, nombre_comun)
                                if os.path.isdir(nombre_comun_path):
                                    # Recorrer los cantos
                                    for canto in os.listdir(nombre_comun_path):
                                        canto_path = os.path.join(nombre_comun_path, canto)
                                        if os.path.isfile(canto_path):
                                            data.append({
                                                'Specie': especie,
                                                'Genus': genero,
                                                'Family': familia,
                                                'Path': canto_path
                                            })

    # Crear el DataFrame
    df = pd.DataFrame(data)
    return df

# Definir la ruta base
base_dir = '/Users/camcortes/Documents/birds-sounds/notebooks/songs'

# Obtener el DataFrame
df = obtener_paths_cantos(base_dir)

# Mostrar el DataFrame
df.sample(4)

,Specie,Genus,Family,Path
79466,Coryphospingus pileatus,Coryphospingus,Thraupidae,/Users/camcortes/Documents/birds-sounds/notebo...
30569,Mitrephanes phaeocercus,Mitrephanes,Tyrannidae,/Users/camcortes/Documents/birds-sounds/notebo...
46695,Myrmelastes leucostigma,Myrmelastes,Thamnophilidae,/Users/camcortes/Documents/birds-sounds/notebo...
20994,Elaenia parvirostris,Elaenia,Tyrannidae,/Users/camcortes/Documents/birds-sounds/notebo...


In [3]:
print(df['Path'][0])

/Users/camcortes/Documents/birds-sounds/notebooks/songs/Capitonidae/Capito/Capito auratus/GildedBarbet/47717.mp3


In [4]:
# guardar el dataframe en un archivo csv
df.to_csv('/Users/camcortes/Documents/birds-sounds/data/paths_cantos_passeriformes.csv', index=False)

In [5]:
familia = taxonomia.capitalize_family_genus(taxonomia.genus_family)
df['Family'] = df['Genus'].map(familia)
df['Suborder'] = df['Family'].map(taxonomia.suborder)
df.sample(5)

,Specie,Genus,Family,Path,Suborder
39911,Pygiptila stellaris,Pygiptila,Thamnophilidae,/Users/camcortes/Documents/birds-sounds/notebo...,tiranni
29911,Phaeomyias murina,Phaeomyias,Tyrannidae,/Users/camcortes/Documents/birds-sounds/notebo...,tiranni
51396,Vireo griseus,Vireo,Vireonidae,/Users/camcortes/Documents/birds-sounds/notebo...,passeri
88568,Manacus manacus,Manacus,Pipridae,/Users/camcortes/Documents/birds-sounds/notebo...,tiranni
30440,Lophotriccus vitiosus,Lophotriccus,Tyrannidae,/Users/camcortes/Documents/birds-sounds/notebo...,tiranni


In [6]:
df['Suborder'].value_counts()

Suborder
tiranni           50631
passeri           37682
Falconiformes       670
Piciformes          658
Psittaciformes      283
Name: count, dtype: int64

In [7]:
species_not_audios = ['Chloropipo flavicapilla', 'Cotinga maynana', 'Lonchura malacca', 'Haematoderus militaris',
    'Psarocolius guatimozinus', 'Siptornis striaticollis', 'Polioptila guianensis', 'Eubucco richardsoni', 'Chlorospingus tacarcunae','Snowornis cryptolophus',
    'Spinus yarrellii', 'Polioptila facilis', 'Chlorospingus inornatus', 'Carpodectes hopkei', 'Cotinga nattererii', 'Tangara fucosa',
    'Bangsia arcaei', 'Gymnoderus foetidus', 'Margarornis bellulus', 'Cotinga cotinga', 'Dacnis viguieri']

In [8]:
#eliminar especies que no tienen audios
df = df[~df['Specie'].isin(species_not_audios)]

In [9]:
df = df.rename(columns={'Specie':'label', 'Path': 'audio_path'})
df = df.reset_index(drop=True)

In [10]:
df.shape

(89872, 5)

In [11]:
from sklearn.model_selection import StratifiedShuffleSplit

X = df.drop(columns=['label'])
y = df['label']

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)

for train_index, test_index in sss.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y[train_index], y[test_index]

In [12]:
X_train.to_csv('/Users/camcortes/Documents/birds-sounds/data/train_specie.csv', index=False)
X_test.to_csv('/Users/camcortes/Documents/birds-sounds/data/test_specie.csv', index=False)